# ReSnap — Retrain v4 (YOLOv8s + data Open Images untuk kelas lemah)

**Masalah v3:** kentang (AP50 val 0.26) dan udang (0.11) hampir selalu terbaca jamur atau bawang putih, bahkan dengan confidence 90%+. Kurva v3 sudah mendatar sejak sekitar epoch 35, jadi menambah epoch tidak membantu. Penyebab utamanya data kelas itu sedikit, ditambah kapasitas YOLOv8n yang kecil.

**Yang diubah di v4** (val tetap sama persis dengan v3 supaya bisa dibandingkan):
1. Foto kentang, udang, dan jamur dari Open Images V7 (berlabel kotak) ditambahkan ke **train saja**.
2. Foto train lama yang memuat kentang/udang disalin 1x lagi (oversampling ringan).
3. Model YOLOv8s pralatih COCO (sekitar 3x parameter YOLOv8n).

**Penjaga kualitas data:** foto "group of" dan ilustrasi dibuang, foto yang mirip foto train/val/test lama (dHash, termasuk versi cermin) dibuang, dan foto yang menurut v3 berisi bahan ReSnap lain tanpa label juga dibuang.

**Penjaga overfit/underfit:**
- Early stopping (patience 20) dan `best.pt` dipilih dari val.
- Augmentasi sama dengan v3, dan oversampling dibatasi 1 salinan.
- Diagnosa setelah training: kurva loss train vs val, gap mAP train-subset vs val dibandingkan gap v3, dan tren mAP di akhir.
- Holdout independen: foto **validation split** Open Images, yang tidak ikut training, untuk menilai kentang/udang/jamur.

Test set (654 foto) **tidak dipakai di sini**. Setelah v4 diterima, jalankan `ReSnap_Evaluasi_Final_Test.ipynb` sekali saja.

Runtime: **GPU T4**. Training kira-kira 3–5 jam. Semua checkpoint disimpan ke Drive; kalau sesi putus, jalankan ulang bagian 1, lalu set `RESUME = True` di bagian 5.

## 1. Paket dan konfigurasi

In [ ]:
%pip -q install ultralytics==8.3.228 fiftyone

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path

PROJECT = Path('/content/drive/MyDrive/Brawijaya/MATKUL/SEM 5/COMVIS/Tugas Akhir')
WORK = Path('/content/resnap_v4')
DEVICE = '0'

In [ ]:
import json, random, shutil, sys, zipfile
from collections import Counter

REPO = PROJECT / 'resnap'
sys.path.insert(0, str(REPO / 'scripts'))
import data_v4 as dv
from diagnose_fit import analyze_curves, load_results, plot as plot_curves, verdict
from evaluate_export import evaluate

CANDIDATE = PROJECT / 'review_anotasi/tambahan_foto_udang_20261120_225843_394191/ReSnap_Kandidat_Tambah_Foto_Udang.zip'
CANDIDATE_SHA = '58d62af77338faa546fff2a77d0bc102b24956dc13fc8065a619dc7145b8e04c'
V3_WEIGHTS = PROJECT / 'training_yolov8/resnap_yolov8n_30kelas_v3_udang_b1/weights/best.pt'
V3_SHA = 'ac24d434cc97d12f3fd3ed204d2fefda8cccfc322422a8c55672190a0615da98'
RUN_PROJECT = PROJECT / 'training_yolov8'
RUN_NAME = 'resnap_yolov8s_30kelas_v4_oi'
RUN_DIR = RUN_PROJECT / RUN_NAME
OUT = PROJECT / 'evaluasi_v4'            # log data, diagnosa, perbandingan
N_TRAIN, N_VAL, N_TEST = 4500, 1069, 654

# Kuota foto Open Images yang DIMINTA (sebelum disaring; biasanya 40-70% yang lolos).
OI_TRAIN_REQUEST = {'Potato': 700, 'Shrimp': 700, 'Mushroom': 350}
OI_HOLDOUT_REQUEST = {'Potato': 200, 'Shrimp': 200, 'Mushroom': 150}
WEAK = {7, 29}                           # potato, shrimp -> dioversample
FOCUS = {7, 13, 29}                      # kelas yang diperbaiki (diabaikan oleh filter v3)
OVERSAMPLE_COPIES = 1

for p in (REPO / 'scripts/data_v4.py', CANDIDATE, V3_WEIGHTS):
    assert p.exists(), f'Tidak ditemukan: {p}'
assert dv.sha256_file(V3_WEIGHTS) == V3_SHA, 'Bobot v3 tidak cocok'
WORK.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)
print('Konfigurasi OK')

## 2. Ekstrak dataset v3 dan hitung hash referensi

Test hanya diekstrak untuk di-hash (mencegah foto Open Images yang mirip foto test masuk train). Test tidak masuk `data.yaml`.

In [ ]:
base = WORK / 'base'
local_zip = WORK / 'candidate.zip'
if not local_zip.exists() or dv.sha256_file(local_zip) != CANDIDATE_SHA:
    shutil.copyfile(CANDIDATE, local_zip)
assert dv.sha256_file(local_zip) == CANDIDATE_SHA, 'Checksum ZIP kandidat tidak cocok'
if not (base / 'train').exists():
    with zipfile.ZipFile(local_zip) as z:
        z.extractall(base, [n for n in z.namelist() if n.startswith(('train/', 'val/', 'test/'))])
counts = {s: len(list((base / s / 'images').iterdir())) for s in ('train', 'val', 'test')}
print(counts)
assert counts == {'train': N_TRAIN, 'val': N_VAL, 'test': N_TEST}, counts

ref_hashes = {}
for split in ('train', 'val', 'test'):
    ref_hashes.update(dv.hash_folder(base / split / 'images'))
print(f'{len(ref_hashes)} foto lama di-hash')

## 3. Unduh Open Images V7

Pertama kali, FiftyOne mengunduh metadata anotasi (beberapa GB, sekitar 5–15 menit). Foto train diambil dari split `train`, holdout dari split `validation` sehingga keduanya tidak tumpang tindih.

In [ ]:
import fiftyone.zoo as foz
import fiftyone as fo

def flag(det, name):
    try:
        return bool(det[name])
    except (KeyError, AttributeError):
        return False

def to_plain(dataset):
    field = next(n for n, f in dataset.get_field_schema().items()
                 if getattr(f, 'document_type', None) is fo.Detections)
    out = []
    for s in dataset:
        dets = s[field].detections if s[field] else []
        out.append({'id': s.open_images_id if s.has_field('open_images_id') else s.id,
                    'filepath': s.filepath,
                    'detections': [{'label': d.label, 'bbox': list(d.bounding_box),
                                    'group': flag(d, 'IsGroupOf'), 'depiction': flag(d, 'IsDepiction')}
                                   for d in dets]})
    return out

def load_oi(split, request):
    seen, samples = set(), []
    for cls, n in request.items():
        ds = foz.load_zoo_dataset('open-images-v7', split=split, label_types=['detections'],
                                  classes=[cls], max_samples=n, shuffle=True, seed=42,
                                  dataset_name=f'resnap_oi_{split}_{cls}',
                                  drop_existing_dataset=True)
        for s in to_plain(ds):
            if s['id'] not in seen:
                seen.add(s['id'])
                samples.append(s)
    return samples

oi_train = load_oi('train', OI_TRAIN_REQUEST)
oi_holdout = load_oi('validation', OI_HOLDOUT_REQUEST)
print(len(oi_train), 'kandidat train,', len(oi_holdout), 'kandidat holdout')
example = next(d for s in oi_train for d in s['detections'])
print('Contoh deteksi:', example)
n_group = sum(d['group'] for s in oi_train for d in s['detections'])
print('Kotak group-of terbaca:', n_group)
if n_group == 0:
    print('PERINGATAN: tidak ada kotak group-of sama sekali. Kemungkinan nama atribut berbeda di versi FiftyOne ini;'
          ' jalankan  print(ds.first().ground_truth.detections[0])  dan sesuaikan nama di fungsi flag().')

## 4. Saring, gabung, dan oversample

In [ ]:
from ultralytics import YOLO

v3 = YOLO(str(V3_WEIGHTS))
data = WORK / 'data'
if data.exists():
    shutil.rmtree(data)
shutil.copytree(base / 'train', data / 'train')
shutil.copytree(base / 'val', data / 'val')

targets = {'Potato', 'Shrimp', 'Mushroom'}
log_train = dv.build_oi_split(oi_train, targets, data / 'train', 'oi', ref_hashes,
                              model=v3, ignore_cls=FOCUS)
kept = {r['oi_id'] for r in log_train if r['status'] == 'dipakai'}
train_oi_hashes = dv.hash_folder(data / 'train' / 'images')
train_oi_hashes = {k: v for k, v in train_oi_hashes.items() if Path(k).stem.startswith('oi_')}
log_hold = dv.build_oi_split(oi_holdout, targets, data / 'holdout_oi', 'oiv', {**ref_hashes, **train_oi_hashes},
                             model=v3, ignore_cls=FOCUS)
added = dv.oversample(data / 'train', WEAK, OVERSAMPLE_COPIES)

for name, log in (('train', log_train), ('holdout', log_hold)):
    c = Counter(r['status'] for r in log)
    print(name, dict(c))
    unl = Counter(x for r in log if r['status'] == 'bahan_lain_tanpa_label' for x in r['detail'].split(';'))
    if unl:
        print('  bahan tanpa label menurut v3:', dict(unl.most_common(8)))
    if c['bahan_lain_tanpa_label'] > 0.4 * max(1, len(log)):
        print('  PERINGATAN: >40% dibuang karena bahan lain tanpa label. Cek kelas di atas;'
              ' kalau didominasi garlic/chicken pada foto udang, itu salah deteksi v3, bukan label hilang.')
print('salinan oversampling:', added)

dv.write_log(OUT / 'log_openimages_train.csv', log_train)
dv.write_log(OUT / 'log_openimages_holdout.csv', log_hold)
stats = {'train_v3': dv.count_split(base / 'train'), 'train_v4': dv.count_split(data / 'train'),
         'val': dv.count_split(data / 'val'), 'holdout_oi': dv.count_split(data / 'holdout_oi')}
dv.save_json(OUT / 'jumlah_data.json', stats)
for cls in ('potato', 'shrimp', 'mushroom'):
    a, b = stats['train_v3']['per_class'][cls], stats['train_v4']['per_class'][cls]
    h = stats['holdout_oi']['per_class'][cls]
    print(f"{cls:9s} train foto {a['images']:4d} -> {b['images']:4d} | kotak {a['instances']:5d} -> {b['instances']:5d}"
          f" | holdout foto {h['images']}")
assert stats['holdout_oi']['images'] > 0, 'Holdout kosong'

yaml_train = dv.write_yaml(data / 'data.yaml', data, 'train/images', 'val/images')
yaml_holdout = dv.write_yaml(data / 'holdout.yaml', data, 'holdout_oi/images', 'holdout_oi/images')

Periksa sekilas 12 foto Open Images yang lolos. Kalau banyak kotak yang meleset atau ada bahan yang jelas tidak berlabel, hentikan dulu sebelum training.

In [ ]:
import cv2, matplotlib.pyplot as plt
oi_imgs = sorted(p for p in (data / 'train/images').iterdir() if p.stem.startswith('oi_'))
random.Random(0).shuffle(oi_imgs)
fig, axes = plt.subplots(3, 4, figsize=(16, 11))
for ax, p in zip(axes.flat, oi_imgs[:12]):
    img = cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    for ln in (data / 'train/labels' / f'{p.stem}.txt').read_text().splitlines():
        c, cx, cy, bw, bh = ln.split(); c = int(c)
        x1, y1 = int((float(cx) - float(bw) / 2) * w), int((float(cy) - float(bh) / 2) * h)
        x2, y2 = int((float(cx) + float(bw) / 2) * w), int((float(cy) + float(bh) / 2) * h)
        cv2.rectangle(img, (x1, y1), (x2, y2), (255, 0, 0), max(2, w // 300))
        cv2.putText(img, dv.CLASS_NAMES[c], (x1, max(15, y1 - 5)), cv2.FONT_HERSHEY_SIMPLEX, w / 900, (255, 0, 0), max(1, w // 500))
    ax.imshow(img); ax.set_title(p.stem[:28], fontsize=8); ax.axis('off')
plt.tight_layout(); plt.savefig(OUT / 'contoh_openimages.png', dpi=100); plt.show()

### Simpan dataset lalu restart session

Unduhan Open Images lewat FiftyOne memakan RAM beberapa GB dan tidak dilepas, sehingga training di sesi yang sama bisa membuat Colab crash. Sel ini menyimpan dataset v4 ke Drive. Setelah selesai: **Runtime > Restart session**, lalu jalankan sel mount + konfigurasi (bagian 1, tanpa `%pip`), bagian 2, dan sel *Muat ulang dataset* di bawah. Bagian 3–4 tidak perlu diulang.

In [ ]:
DATASET_ZIP = PROJECT / 'dataset_v4.zip'
tmp_zip = shutil.make_archive(str(WORK / 'dataset_v4'), 'zip', root_dir=data)
shutil.copyfile(tmp_zip, DATASET_ZIP)
print('Tersimpan:', DATASET_ZIP, round(DATASET_ZIP.stat().st_size / 1e6), 'MB')
print('Sekarang: Runtime > Restart session.')

### Muat ulang dataset (setelah restart)

In [ ]:
DATASET_ZIP = PROJECT / 'dataset_v4.zip'
data = WORK / 'data'
if not (data / 'train').exists():
    local = WORK / 'dataset_v4.zip'
    shutil.copyfile(DATASET_ZIP, local)
    with zipfile.ZipFile(local) as z:
        z.extractall(data)
    local.unlink()
stats = json.loads((OUT / 'jumlah_data.json').read_text(encoding='utf-8'))
found = {s: len(list((data / s / 'images').iterdir())) for s in ('train', 'val', 'holdout_oi')}
print(found)
assert found == {'train': stats['train_v4']['images'], 'val': stats['val']['images'],
                 'holdout_oi': stats['holdout_oi']['images']}, 'Isi dataset tidak cocok dengan jumlah_data.json'
yaml_train = dv.write_yaml(data / 'data.yaml', data, 'train/images', 'val/images')
yaml_holdout = dv.write_yaml(data / 'holdout.yaml', data, 'holdout_oi/images', 'holdout_oi/images')
from ultralytics import YOLO
print('Dataset siap untuk training')

## 5. Training YOLOv8s

Setelan augmentasi sama dengan v3. Optimizer dibuat eksplisit `AdamW` dengan lr 0.0003, karena itulah yang sebenarnya dipakai v3 lewat `optimizer='auto'` (nilai `lr0=0.01` di `experiment.json` v3 diabaikan Ultralytics). Batas 100 epoch dengan patience 20: model berhenti sendiri kalau val tidak membaik 20 epoch, dan `best.pt` diambil dari epoch dengan val terbaik.

In [ ]:
RESUME = False   # True jika sesi terputus dan RUN_DIR/weights/last.pt sudah ada
MAX_EPOCHS = 100
RECIPE = dict(
    epochs=MAX_EPOCHS, patience=20, batch=16, imgsz=640, save=True, save_period=5, cache=False,
    workers=2, pretrained=True, optimizer='AdamW', lr0=0.0003, lrf=0.01, momentum=0.937,
    weight_decay=0.0005, warmup_epochs=3.0, seed=42, deterministic=True, cos_lr=False,
    close_mosaic=10, amp=True, plots=True, val=True,
    hsv_h=0.015, hsv_s=0.5, hsv_v=0.35, degrees=10.0, translate=0.1, scale=0.4, shear=0.0,
    perspective=0.0, flipud=0.0, fliplr=0.5, mosaic=0.5, mixup=0.0, copy_paste=0.0,
)
if RESUME:
    YOLO(str(RUN_DIR / 'weights/last.pt')).train(resume=True, device=DEVICE)
else:
    assert not RUN_DIR.exists(), f'{RUN_DIR} sudah ada. Ganti RUN_NAME atau set RESUME=True.'
    RUN_DIR.mkdir(parents=True)
    dv.save_json(RUN_DIR / 'experiment.json', {'base_model': 'yolov8s.pt', 'settings': RECIPE,
                                               'oversample_copies': OVERSAMPLE_COPIES,
                                               'oi_train_request': OI_TRAIN_REQUEST,
                                               'train_images': stats['train_v4']['images']})
    YOLO('yolov8s.pt').train(data=str(yaml_train), project=str(RUN_PROJECT), name=RUN_NAME,
                             exist_ok=True, device=DEVICE, **RECIPE)
V4_WEIGHTS = RUN_DIR / 'weights/best.pt'
dv.save_json(RUN_DIR / 'training_complete.json', {'weights_sha256': dv.sha256_file(V4_WEIGHTS)})
print('Selesai:', V4_WEIGHTS)

## 6. Diagnosa overfit / underfit

Gap dihitung dari mAP50-95 pada 600 foto train (tanpa augmentasi, bukan salinan oversampling) dikurangi mAP50-95 val. Pembandingnya gap v3 yang diukur dengan cara yang sama, karena detektor wajar punya gap.

In [ ]:
V4_WEIGHTS = RUN_DIR / 'weights/best.pt'
rng = random.Random(42)

def subset_yaml(src_dir, n, name):
    imgs = [p for p in (src_dir / 'images').iterdir() if '_dup' not in p.stem]
    pick = rng.sample(imgs, min(n, len(imgs)))
    dst = WORK / name
    if dst.exists():
        shutil.rmtree(dst)
    (dst / 'images').mkdir(parents=True); (dst / 'labels').mkdir()
    for p in pick:
        shutil.copy2(p, dst / 'images' / p.name)
        shutil.copy2(src_dir / 'labels' / f'{p.stem}.txt', dst / 'labels' / f'{p.stem}.txt')
    return dv.write_yaml(dst / 'data.yaml', dst, 'images', 'images')

yaml_tr_v4 = subset_yaml(data / 'train', 600, 'train_subset_v4')
yaml_tr_v3 = subset_yaml(base / 'train', 600, 'train_subset_v3')

E = {}
E['v4_val'] = evaluate(str(V4_WEIGHTS), yaml_train, 'val', OUT, 'v4_val', DEVICE)
E['v4_train'] = evaluate(str(V4_WEIGHTS), yaml_tr_v4, 'val', OUT, 'v4_train_subset', DEVICE)
E['v3_val'] = evaluate(str(V3_WEIGHTS), yaml_train, 'val', OUT, 'v3_val', DEVICE)
E['v3_train'] = evaluate(str(V3_WEIGHTS), yaml_tr_v3, 'val', OUT, 'v3_train_subset', DEVICE)
E['v4_holdout'] = evaluate(str(V4_WEIGHTS), yaml_holdout, 'val', OUT, 'v4_holdout', DEVICE)
E['v3_holdout'] = evaluate(str(V3_WEIGHTS), yaml_holdout, 'val', OUT, 'v3_holdout', DEVICE)

ref_gap = E['v3_train']['mAP50_95'] - E['v3_val']['mAP50_95']
rows = load_results(RUN_DIR / 'results.csv')
curves = analyze_curves(rows, MAX_EPOCHS)
fit = verdict(curves, E['v4_train']['mAP50_95'], E['v4_val']['mAP50_95'], ref_gap)
plot_curves(rows, OUT / 'kurva_v4.png', curves['best_epoch'], 'Kurva training v4 (YOLOv8s)')
dv.save_json(OUT / 'diagnosa_fit.json', {'kurva': curves, 'gap_v3': ref_gap, **fit, 'evaluasi': E})
print(json.dumps({'kurva': curves, 'gap_v3': round(ref_gap, 4), **fit}, indent=2, ensure_ascii=False))
from IPython.display import Image, display
display(Image(str(OUT / 'kurva_v4.png'), width=900))

## 7. Bandingkan v3 dan v4

Val dipakai untuk semua kelas. Holdout Open Images hanya dibaca untuk kentang, udang, dan jamur, karena foto itu bisa memuat bahan lain yang tidak berlabel.

In [ ]:
import pandas as pd

def per_class(name):
    return pd.read_csv(OUT / name / 'per_class.csv').set_index('class_name')

overall = pd.DataFrame({k: E[k] for k in ('v3_val', 'v4_val')}).loc[
    ['precision', 'recall', 'f1', 'mAP50', 'mAP75', 'mAP50_95']]
overall['delta_pp'] = 100 * (overall['v4_val'] - overall['v3_val'])
display(overall.style.format(precision=3))

pv3, pv4 = per_class('v3_val'), per_class('v4_val')
cmp = pd.DataFrame({'instances': pv4['instances'], 'AP50_v3': pv3['AP50'], 'AP50_v4': pv4['AP50']})
cmp['delta_pp'] = 100 * (cmp['AP50_v4'] - cmp['AP50_v3'])
display(cmp.sort_values('delta_pp').style.format(precision=3).set_caption('AP50 per kelas di val'))

hv3, hv4 = per_class('v3_holdout'), per_class('v4_holdout')
focus = [c for c in ('potato', 'shrimp', 'mushroom') if c in hv4.index]
hold = pd.DataFrame({'instances': hv4.loc[focus, 'instances'],
                     'AP50_v3': hv3.reindex(focus)['AP50'].fillna(0), 'AP50_v4': hv4.loc[focus, 'AP50'],
                     'recall_v3': hv3.reindex(focus)['recall'].fillna(0), 'recall_v4': hv4.loc[focus, 'recall']})
hold['delta_AP50_pp'] = 100 * (hold['AP50_v4'] - hold['AP50_v3'])
display(hold.style.format(precision=3).set_caption('Holdout Open Images (tidak ikut training)'))
cmp.to_csv(OUT / 'banding_per_kelas_val.csv'); hold.to_csv(OUT / 'banding_holdout.csv')

## 8. Keputusan

v4 diterima bila **semua** syarat terpenuhi:
1. Vonis diagnosa `seimbang` (tidak overfit dan sudah konvergen).
2. mAP50-95 val tidak turun lebih dari 0.5 poin dari v3.
3. AP50 holdout naik minimal 5 poin untuk kentang **dan** udang.
4. Tidak ada kelas val dengan minimal 20 instance yang AP50-nya turun lebih dari 10 poin.

Kalau gagal, petunjuk perbaikannya dicetak di bawah.

In [ ]:
checks = {
    'fit_seimbang': fit['vonis'] == 'seimbang',
    'val_map_tidak_turun': overall.loc['mAP50_95', 'delta_pp'] >= -0.5,
    'kentang_naik': 'potato' in hold.index and hold.loc['potato', 'delta_AP50_pp'] >= 5,
    'udang_naik': 'shrimp' in hold.index and hold.loc['shrimp', 'delta_AP50_pp'] >= 5,
    'tidak_ada_kelas_anjlok': not ((cmp['instances'] >= 20) & (cmp['delta_pp'] < -10)).any(),
}
ACCEPT = all(checks.values())
for k, v in checks.items():
    print(('LULUS ' if v else 'GAGAL ') + k)
tips = {
    'fit_seimbang': {
        'underfit': 'Belum konvergen: training ulang dengan MAX_EPOCHS lebih besar (mis. 150) dan RUN_NAME baru. Resume tidak bisa menambah epoch.',
        'overfit': 'Overfit: OVERSAMPLE_COPIES=0 dan mosaic=1.0, lalu training ulang dengan RUN_NAME baru. best.pt saat ini tetap dari epoch val terbaik.',
    }.get(fit['vonis'], 'Overfit sekaligus belum konvergen: kurangi oversampling (OVERSAMPLE_COPIES=0) dan naikkan MAX_EPOCHS, RUN_NAME baru.'),
    'val_map_tidak_turun': 'Data Open Images menggeser distribusi. Kurangi OI_TRAIN_REQUEST Mushroom atau cek contoh_openimages.png.',
    'kentang_naik': 'Tambah OI_TRAIN_REQUEST Potato (mis. 1200) dan periksa log_openimages_train.csv.',
    'udang_naik': 'Tambah OI_TRAIN_REQUEST Shrimp (mis. 1200) dan periksa log_openimages_train.csv.',
    'tidak_ada_kelas_anjlok': 'Lihat tabel per kelas; kelas yang anjlok mungkin tertutup label Open Images yang hilang.',
}
for k, v in checks.items():
    if not v:
        print('->', tips[k])
dv.save_json(OUT / 'keputusan.json', {'accept': ACCEPT, 'checks': {k: bool(v) for k, v in checks.items()}})
print('\nv4 DITERIMA' if ACCEPT else '\nv4 BELUM DITERIMA, v3 tetap dipakai')

Kalau diterima, salin ke repo. `app.py` otomatis memakai `models/best.pt`; v3 disimpan sebagai `best_v3.pt`. Setelah itu jalankan `ReSnap_Evaluasi_Final_Test.ipynb` dengan `MODEL = 'v4'`.

In [ ]:
if ACCEPT:
    models_dir = REPO / 'models'
    if not (models_dir / 'best_v3.pt').exists():
        shutil.copy2(models_dir / 'best.pt', models_dir / 'best_v3.pt')
    shutil.copy2(V4_WEIGHTS, models_dir / 'best.pt')
    (models_dir / 'best.onnx').unlink(missing_ok=True)  # ONNX lama milik v3; buat ulang lewat notebook evaluasi final
    print('models/best.pt sekarang v4:', dv.sha256_file(models_dir / 'best.pt'))
else:
    print('Tidak ada yang disalin.')